In [ ]:
#!pip install pdfminer
#!pip install PyPDF2
#!pip install pdfplumber

In [4]:
import pandas as pd
import os
import json
import numpy as np
from pdfminer.high_level import extract_text
from PyPDF2 import PdfReader
import pdfplumber

### Manipulação do relatório CSV

In [ ]:
relatorio = pd.read_csv(r"C:\Users\Usuario\Desktop\Repositorios_Bitbucket\crawer-ri-gpa\relatorios_consolidados.csv")
df_csv = pd.DataFrame(relatorio)

In [ ]:
df_filtrado = df_csv[~df_csv["empresa"].str.contains("MAGALU", case=False, na=False)]

In [10]:
# Caminho da pasta com os JSONs da Magalu
caminho_json_magalu = r"C:\Users\Usuario\Desktop\Repositorios_Bitbucket\crawer-ri-gpa\pdfs\Magalu"

# Lista para armazenar os dados
dados_json_magalu = []

# Percorre os arquivos da pasta
for raiz, _, arquivos in os.walk(caminho_json_magalu):
    if 'jsons' in raiz.lower():
        for nome_arquivo in arquivos:
            if nome_arquivo.endswith(".json"):
                caminho_json = os.path.join(raiz, nome_arquivo)
                try:
                    with open(caminho_json, "r", encoding="utf-8") as f:
                        dados = json.load(f)

                    dados_json_magalu.append({
                        "empresa": dados.get("empresa", "Desconhecida"),
                        "categoria": dados.get("categoria", ""),
                        "ano": dados.get("ano", ""),
                        "trimestre": dados.get("trimestre", ""),
                        "url": dados.get("download_url", ""),
                        "arquivo": nome_arquivo,
                        "paginas": dados.get("num_pages", ""),
                        "data_extracao": dados.get("data_extracao", ""),
                        "checksum": dados.get("checksum", "")
                    })
                except Exception as e:
                    print(f"Erro ao processar {caminho_json}: {e}")

# Cria DataFrame com os dados extraídos
df_json_magalu = pd.DataFrame(dados_json_magalu)

# Garante que todas as colunas estejam presentes
colunas_esperadas = ["empresa", "categoria", "ano", "trimestre", "url", "arquivo", "paginas", "data_extracao", "checksum"]
for col in colunas_esperadas:
    if col not in df_json_magalu.columns:
        df_json_magalu[col] = ""

df_json_magalu = df_json_magalu[colunas_esperadas]

# Carrega o CSV existente (se houver)
try:
    df_existente = pd.read_csv("relatorio_consolidado.csv")
except FileNotFoundError:
    df_existente = pd.DataFrame(columns=colunas_esperadas)

# Concatena os dados
df_final = pd.concat([df_existente, df_json_magalu], ignore_index=True)

# Salva o resultado atualizado
df_final.to_csv("relatorio_consolidado.csv", index=False)

print("Dados da Magalu adicionados com sucesso ao relatorio_consolidado.csv.")


Dados da Magalu adicionados com sucesso ao relatorio_consolidado.csv.


### Avaliação da raspagem

In [6]:
def verificar_relatorios_em_empresas(diretorio_base="pdfs"):
    print(f"\n Verificando relatórios na pasta base: {diretorio_base}\n")

    for empresa in os.listdir(diretorio_base):
        caminho_empresa = os.path.join(diretorio_base, empresa)
        if not os.path.isdir(caminho_empresa):
            continue

        print(f"\n Empresa: {empresa}")

        for pasta_ano in os.listdir(caminho_empresa):
            caminho_ano = os.path.join(caminho_empresa, pasta_ano)

            if os.path.isdir(caminho_ano):
                caminho_originais = os.path.join(caminho_ano, "originais")

                if os.path.exists(caminho_originais):
                    arquivos_pdf = [
                        f for f in os.listdir(caminho_originais)
                        if f.lower().endswith(".pdf")
                    ]
                    total = len(arquivos_pdf)

                    if total < 12:
                        print(f"{pasta_ano} - Apenas {total} relatórios encontrados (esperado: 12)")
                    else:
                        print(f"{pasta_ano} - OK ({total} relatórios)")
                else:
                    print(f"{pasta_ano} - Pasta 'originais' não encontrada")

verificar_relatorios_em_empresas("C:/Users/Usuario/Desktop/Repositorios_Bitbucket/crawer-ri-gpa/pdfs")


 Verificando relatórios na pasta base: C:/Users/Usuario/Desktop/Repositorios_Bitbucket/crawer-ri-gpa/pdfs


 Empresa: AmericanasSA
2020 - OK (13 relatórios)
2021 - OK (13 relatórios)
2022 - OK (12 relatórios)
2023 - OK (12 relatórios)
2024 - OK (12 relatórios)
2025 - Apenas 3 relatórios encontrados (esperado: 12)

 Empresa: CasasBahia
2021 - OK (12 relatórios)
2022 - OK (12 relatórios)
2023 - OK (12 relatórios)
2024 - OK (12 relatórios)
2025 - Apenas 3 relatórios encontrados (esperado: 12)

 Empresa: GPA
2020 - OK (12 relatórios)
2021 - OK (12 relatórios)
2022 - OK (12 relatórios)
2023 - OK (12 relatórios)
2024 - OK (12 relatórios)
2025 - Apenas 3 relatórios encontrados (esperado: 12)

 Empresa: MAGALU
2021 - OK (16 relatórios)
2022 - OK (16 relatórios)
2023 - OK (16 relatórios)
2024 - OK (16 relatórios)
2025 - Apenas 5 relatórios encontrados (esperado: 12)

 Empresa: PagueMenos
2020 - Apenas 10 relatórios encontrados (esperado: 12)
2021 - OK (12 relatórios)
2022 - OK (12 relatórios)


### Avaliação da Saida CSV

In [7]:
dados_csv = pd.read_csv(r"C:\Users\Usuario\Desktop\Repositorios_Bitbucket\crawer-ri-gpa\relatorios_consolidados.csv")
df_csv = pd.DataFrame(dados_csv)

In [8]:
df_csv.head()

,empresa,categoria,ano,trimestre,url,arquivo,paginas,data_extracao,checksum
0,MAGALU,Outros,2025,1T25,https://ri.magazineluiza.com.br/Download.aspx?...,pdfs\MAGALU\2025\originais\2025_1T25_RL.pdf,27,2025-05-21T16:16:08.952553+00:00,a67a7131a3e63cca0e77272ff8093b3d89adf84ed7a453...
1,MAGALU,Demonstrações Financeiras,2025,1T25,https://ri.magazineluiza.com.br/Download.aspx?...,pdfs\MAGALU\2025\originais\2025_1T25_ITR.pdf,57,2025-05-21T16:16:09.597321+00:00,d242627adaf34683d899dcf1f4a593a82df8bf7d4a07b3...
2,MAGALU,Apresentações,2025,1T25,https://ri.magazineluiza.com.br/Download.aspx?...,pdfs\MAGALU\2025\originais\2025_1T25_AP.pdf,24,2025-05-21T16:16:10.153871+00:00,ad0450985e0be0b36ecf2468ad0ad799405f1613568d1d...
3,MAGALU,Desconhecido,2025,1T25,https://ri.magazineluiza.com.br/Download.aspx?...,pdfs\MAGALU\2025\originais\2025_1T25_UNK.pdf,0,2025-05-21T16:16:39.243885+00:00,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...
4,MAGALU,Desconhecido,2025,1T25,https://ri.magazineluiza.com.br/Download.aspx?...,pdfs\MAGALU\2025\originais\2025_1T25_UNK.pdf,0,2025-05-21T16:16:39.754894+00:00,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...


In [9]:
#Zeros por coluna numerica
contagem_zeros = (df_csv == 0).sum()
print("Contagem de zeros por coluna numérica:", contagem_zeros)

Contagem de zeros por coluna numérica: empresa           0
categoria         0
ano               0
trimestre         0
url               0
arquivo           0
paginas          54
data_extracao     0
checksum          0
dtype: int64


In [10]:
# Filtra as linhas onde a coluna tem valor 0 ou está vazia
linhas_com_zeros_ou_vazias = df_csv[
    (df_csv['paginas'] == 0) | 
    (df_csv['paginas'].isna()) | 
    (df_csv['paginas'] == '')
]

# Exibe as linhas filtradas
linhas_com_zeros_ou_vazias.head()

,empresa,categoria,ano,trimestre,url,arquivo,paginas,data_extracao,checksum
3,MAGALU,Desconhecido,2025,1T25,https://ri.magazineluiza.com.br/Download.aspx?...,pdfs\MAGALU\2025\originais\2025_1T25_UNK.pdf,0,2025-05-21T16:16:39.243885+00:00,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...
4,MAGALU,Desconhecido,2025,1T25,https://ri.magazineluiza.com.br/Download.aspx?...,pdfs\MAGALU\2025\originais\2025_1T25_UNK.pdf,0,2025-05-21T16:16:39.754894+00:00,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...
5,MAGALU,Desconhecido,2025,1T25,https://ri.magazineluiza.com.br/ShowResultado....,pdfs\MAGALU\2025\originais\2025_1T25_UNK.pdf,0,2025-05-21T16:16:39.897145+00:00,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...
6,MAGALU,Desconhecido,2025,NaN,https://ri.magazineluiza.com.br/javascript:voi...,pdfs\MAGALU\2025\originais\2025__UNK.pdf,0,2025-05-21T16:16:40.136581+00:00,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...
7,MAGALU,Desconhecido,2025,NaN,https://ri.magazineluiza.com.br/javascript:voi...,pdfs\MAGALU\2025\originais\2025__UNK.pdf,0,2025-05-21T16:16:40.369217+00:00,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...


In [11]:
#Zeros por coluna tipo string
vazios_str = (df_csv == "").sum()
print("Contagem de valores vazios em formato str", vazios_str)

Contagem de valores vazios em formato str empresa          0
categoria        0
ano              0
trimestre        0
url              0
arquivo          0
paginas          0
data_extracao    0
checksum         0
dtype: int64


### Avaliação da saida de texto no JSON

In [ ]:
import os
import json
import numpy as np


def verificar_jsons_e_raw_text_em_empresas_problemas_apenas(diretorio_base="pdfs"):
    """
    Verifica arquivos JSON e o status de 'raw_text' dentro das estruturas
    de pastas de empresas e anos, retornando apenas informações sobre
    empresas/anos que possuem problemas.

    Problemas incluem:
    - Pasta 'jsons' não encontrada.
    - Nenhum arquivo JSON encontrado na pasta 'jsons'.
    - Arquivos JSON com 'raw_text' vazio ou ausente.
    - Arquivos JSON com 'raw_text' com menos de 100 caracteres.
    - Erros de leitura/parsing de arquivos JSON.

    Args:
        diretorio_base (str): O caminho para a pasta base que contém as pastas
                              das empresas (ex: 'pdfs').
    """
    print(f"\n--- Verificando arquivos JSON e 'raw_text' na pasta base: {diretorio_base} (Apenas Problemas) ---\n")

    # Percorre as pastas de cada empresa dentro do diretório base
    for nome_empresa in os.listdir(diretorio_base):
        caminho_empresa = os.path.join(diretorio_base, nome_empresa)

        # Pula se não for um diretório (ex: arquivos no nível raiz)
        if not os.path.isdir(caminho_empresa):
            continue

        # Lista para armazenar as mensagens de problemas de cada ano da empresa atual
        mensagens_problemas_empresa = []

        # Percorre as pastas de cada ano dentro da pasta da empresa
        for nome_pasta_ano in os.listdir(caminho_empresa):
            caminho_ano = os.path.join(caminho_empresa, nome_pasta_ano)

            # Pula se não for um diretório (ex: arquivos no nível da empresa)
            if not os.path.isdir(caminho_ano):
                continue

            # Flag para indicar se o ano atual tem algum problema
            ano_tem_problema = False
            # Lista temporária para as mensagens deste ano
            mensagens_ano_atual = []

            caminho_jsons = os.path.join(caminho_ano, "jsons")

            if os.path.exists(caminho_jsons) and os.path.isdir(caminho_jsons):
                arquivos_json = [f for f in os.listdir(caminho_jsons) if f.lower().endswith(".json")]
                total_jsons = len(arquivos_json)

                count_raw_text_valido = 0       # raw_text existe e tem >= 100 caracteres
                count_raw_text_vazio_ausente = 0 # raw_text é nulo, vazio ou só espaços
                count_raw_text_curto = 0         # raw_text existe, mas tem < 100 caracteres
                count_erros_leitura = 0

                if total_jsons > 0:
                    for nome_arquivo_json in arquivos_json:
                        caminho_completo_json = os.path.join(caminho_jsons, nome_arquivo_json)
                        try:
                            with open(caminho_completo_json, "r", encoding="utf-8") as f:
                                dados_json = json.load(f)

                            raw_text = dados_json.get("raw_text")

                            if not raw_text or not str(raw_text).strip():
                                count_raw_text_vazio_ausente += 1
                            else:
                                # Se não está vazio/ausente, verifica o comprimento
                                if len(raw_text) < 100:
                                    count_raw_text_curto += 1
                                else:
                                    count_raw_text_valido += 1
                        except Exception as e:
                            count_erros_leitura += 1
                            # Opcional: print(f"    [ERRO DETALHADO] {caminho_completo_json}: {e}")

                    # Verifica se houve *qualquer* problema para este ano
                    if count_raw_text_vazio_ausente > 0 or \
                       count_raw_text_curto > 0 or \
                       count_erros_leitura > 0:
                        ano_tem_problema = True
                        mensagens_ano_atual.append(f"  Ano {nome_pasta_ano}: {total_jsons} JSONs encontrados.")
                        # Sempre mostra o total válido, mesmo que seja 0, para contexto
                        mensagens_ano_atual.append(f"    - {count_raw_text_valido} com 'raw_text' válido (>= 100 chars).")
                        if count_raw_text_vazio_ausente > 0:
                            mensagens_ano_atual.append(f"    - **{count_raw_text_vazio_ausente}** com 'raw_text' vazio ou ausente.")
                        if count_raw_text_curto > 0:
                            mensagens_ano_atual.append(f"    - **{count_raw_text_curto}** com 'raw_text' muito curto (< 100 chars).")
                        if count_erros_leitura > 0:
                            mensagens_ano_atual.append(f"    - **{count_erros_leitura}** com erros de leitura/parsing.")
                else: # total_jsons == 0 (nenhum JSON encontrado na pasta 'jsons')
                    ano_tem_problema = True
                    mensagens_ano_atual.append(f"  Ano {nome_pasta_ano}: NENHUM JSON encontrado na pasta 'jsons'.")
            else: # Pasta 'jsons' não encontrada
                ano_tem_problema = True
                mensagens_ano_atual.append(f"  Ano {nome_pasta_ano}: Pasta 'jsons' **não encontrada** ou não é um diretório.")

            # Se este ano teve algum problema, adiciona suas mensagens à lista da empresa
            if ano_tem_problema:
                mensagens_problemas_empresa.extend(mensagens_ano_atual)

        # Após verificar todos os anos de uma empresa, se houver mensagens de problema, imprima
        if mensagens_problemas_empresa:
            print(f"\n### Empresa: {nome_empresa} (Problemas Encontrados) ###")
            for msg in mensagens_problemas_empresa:
                print(msg)

caminho_base_pdfs = r"C:\Users\Usuario\Desktop\Repositorios_Bitbucket\crawer-ri-gpa\pdfs"

verificar_jsons_e_raw_text_em_empresas_problemas_apenas(caminho_base_pdfs)


--- Verificando arquivos JSON e 'raw_text' na pasta base: C:\Users\Usuario\Desktop\Repositorios_Bitbucket\crawer-ri-gpa\pdfs (Apenas Problemas) ---


### Empresa: MAGALU (Problemas Encontrados) ###
  Ano 2021: 16 JSONs encontrados.
    - 12 com 'raw_text' válido (>= 100 chars).
    - **4** com 'raw_text' vazio ou ausente.
  Ano 2022: 16 JSONs encontrados.
    - 12 com 'raw_text' válido (>= 100 chars).
    - **4** com 'raw_text' vazio ou ausente.
  Ano 2023: 16 JSONs encontrados.
    - 12 com 'raw_text' válido (>= 100 chars).
    - **4** com 'raw_text' vazio ou ausente.
  Ano 2024: 16 JSONs encontrados.
    - 12 com 'raw_text' válido (>= 100 chars).
    - **4** com 'raw_text' vazio ou ausente.
  Ano 2025: 5 JSONs encontrados.
    - 3 com 'raw_text' válido (>= 100 chars).
    - **2** com 'raw_text' vazio ou ausente.

### Empresa: PagueMenos (Problemas Encontrados) ###
  Ano 2020: 10 JSONs encontrados.
    - 8 com 'raw_text' válido (>= 100 chars).
    - **2** com 'raw_text' muito curto (

### Avaliação da saida de txt

In [23]:
# Caminho da pasta onde estão as subpastas MAGALU, Casas Bahia, e GPA
caminho_base = r"C:\Users\Usuario\Desktop\Repositorios_Bitbucket\crawer-ri-gpa\pdfs"

# Lista para armazenar os dados
dados_txt = []

# Percorre todas as pastas e subpastas
for raiz, diretorios, arquivos in os.walk(caminho_base):
    # Verifica se está dentro da pasta 'textos' para processar os arquivos
    if 'textos' in raiz:
        for nome_arquivo in arquivos:
            if nome_arquivo.endswith(".txt"):
                caminho_completo = os.path.join(raiz, nome_arquivo)
                try:
                    # Abre o arquivo .txt e calcula o número total de caracteres
                    with open(caminho_completo, "r", encoding="utf-8") as f:
                        texto = f.read()

                    # Armazena o nome do arquivo e a soma dos caracteres
                    dados_txt.append([nome_arquivo, len(texto)])

                except Exception as e:
                    dados_txt.append([nome_arquivo, f"Erro: {e}"])

# Cria o DataFrame a partir da lista
df_txt = pd.DataFrame(dados_txt, columns=["Arquivo", "Total_caracteres"])

# Exibe o DataFrame
print(df_txt)


             Arquivo  Total_caracteres
0     2020_1T_AP.txt              4092
1    2020_1T_ITR.txt            175081
2     2020_1T_RR.txt             58604
3     2020_2T_AP.txt              5579
4    2020_2T_ITR.txt            180254
..               ...               ...
304  2024_4T_DFP.txt            214486
305   2024_4T_RR.txt             58060
306   2025_1T_AP.txt             14296
307  2025_1T_ITR.txt            156086
308   2025_1T_RR.txt             32511

[309 rows x 2 columns]


In [ ]:
vazios_str = df_txt[df_txt['Total_caracteres'] == 0]
print(vazios_str)

               Arquivo  Total_caracteres
182  2021_1T21_UNK.txt                 0
186  2021_2T21_UNK.txt                 0
190  2021_3T21_UNK.txt                 0
194  2021_4T21_UNK.txt                 0
198  2022_1T22_UNK.txt                 0
202  2022_2T22_UNK.txt                 0
206  2022_3T22_UNK.txt                 0
210  2022_4T22_UNK.txt                 0
214  2023_1T23_UNK.txt                 0
218  2023_2T23_UNK.txt                 0
222  2023_3T23_UNK.txt                 0
226  2023_4T23_UNK.txt                 0
230  2024_1T24_UNK.txt                 0
234  2024_2T24_UNK.txt                 0
238  2024_3T24_UNK.txt                 0
242  2024_4T24_UNK.txt                 0
246  2025_1T25_UNK.txt                 0
247      2025__UNK.txt                 0


### Avaliação dos arquivos individualmente

In [ ]:
def diagnosticar_pdf(caminho_pdf):
    print(f"\nAnalisando: {caminho_pdf}")

    # 1. Verifica número de páginas com PyPDF2
    try:
        reader = PdfReader(caminho_pdf)
        num_paginas = len(reader.pages)
        print(f"Número de páginas: {num_paginas}")
    except Exception as e:
        print(f"Erro ao ler número de páginas: {e}")
        return

    # 2. Extrai texto com pdfminer
    try:
        texto = extract_text(caminho_pdf)
        total_caracteres = len(texto.strip())
        print(f"Caracteres extraídos: {total_caracteres}")
    except Exception as e:
        print(f"Erro ao extrair texto: {e}")
        return

In [13]:
caminho_pdf = ""

In [14]:
def visualizar_pdf(path_pdf, preview_chars=500):
    print(f"\n Analisando: {path_pdf}")
    
    try:
        reader = PdfReader(caminho_pdf)
        num_paginas = len(reader.pages)
        print(f"Número de páginas: {num_paginas}")
        texto = extract_text(path_pdf)
        total = len(texto)
        print(f"Total de caracteres extraídos: {total}")

        if total == 0:
            print(" Texto vazio!")
            return
        
        # Visualização: primeiras e últimas partes
        print("\n INÍCIO DO TEXTO:")
        print(texto[:preview_chars])

        print("\n FIM DO TEXTO:")
        print(texto[-preview_chars:])

    except Exception as e:
        print(f"Erro ao extrair texto: {e}")


In [15]:
from pdfminer.high_level import extract_text

def extrair_com_pdfminer(caminho_pdf: str) -> str:
    """
    Extrai texto de um PDF usando pdfminer.
    Melhor para texto bruto, mas sem layout/table preservation.
    """
    try:
        texto = extract_text(caminho_pdf)
        return texto.strip()
    except Exception as e:
        print(f"[pdfminer] Erro ao extrair texto de {caminho_pdf}: {e}")
        return ""

In [16]:
import pdfplumber

def extrair_com_pdfplumber(caminho_pdf: str) -> str:
    """
    Extrai texto de um PDF usando pdfplumber.
    Preserva mais o layout e estrutura de colunas/tabelas.
    """
    texto = ""
    try:
        with pdfplumber.open(caminho_pdf) as pdf:
            for pagina in pdf.pages:
                pagina_texto = pagina.extract_text()
                if pagina_texto:
                    texto += pagina_texto + "\n"
        return texto.strip()
    except Exception as e:
        print(f"[pdfplumber] Erro ao extrair texto de {caminho_pdf}: {e}")
        return ""


In [19]:
caminho = r"C:\Users\Usuario\Desktop\Repositorios_Bitbucket\crawer-ri-gpa\pdfs\PagueMenos\2021\originais\2021_1T_ITR.pdf"

texto_pdfminer = extrair_com_pdfminer(caminho)
texto_pdfplumber = extrair_com_pdfplumber(caminho)


In [20]:
print(texto_pdfminer)

ITR - Informações Trimestrais - 31/03/2021 - EMPREENDIMENTOS PAGUE MENOS S.A.

Versão : 1

Demonstração do Valor Adicionado10Comentário do Desempenho11DMPL - 01/01/2020 à 31/03/20209Declaração dos Diretores sobre o Relatório do Auditor Independente60DMPL - 01/01/2021 à 31/03/20218Relatório da Revisão Especial - Sem Ressalva57Declaração dos Diretores sobre as Demonstrações Financeiras58Notas Explicativas29Pareceres e DeclaraçõesDFs IndividuaisBalanço Patrimonial Ativo2Composição do Capital1Demonstração das Mutações do Patrimônio LíquidoDados da EmpresaDemonstração do Resultado Abrangente5Demonstração do Fluxo de Caixa6Balanço Patrimonial Passivo3Demonstração do Resultado4ÍndiceITR - Informações Trimestrais - 31/03/2021 - EMPREENDIMENTOS PAGUE MENOS S.A.

Versão : 1

PÁGINA: 1 de 61

Em TesourariaTotal443.781.062Preferenciais0Ordinárias1.040.000Total1.040.000Preferenciais0Do Capital IntegralizadoOrdinárias443.781.062Dados da Empresa / Composição do CapitalNúmero de Ações(cid:10)(Unidade

In [21]:
print(texto_pdfplumber)

ITR - Informações Trimestrais - 31/03/2021 - EMPREENDIMENTOS PAGUE MENOS S.A. Versão : 1
Índice
Dados da Empresa
Composição do Capital 1
DFs Individuais
Balanço Patrimonial Ativo 2
Balanço Patrimonial Passivo 3
Demonstração do Resultado 4
Demonstração do Resultado Abrangente 5
Demonstração do Fluxo de Caixa 6
Demonstração das Mutações do Patrimônio Líquido
DMPL - 01/01/2021 à 31/03/2021 8
DMPL - 01/01/2020 à 31/03/2020 9
Demonstração do Valor Adicionado 10
Comentário do Desempenho 11
Notas Explicativas 29
Pareceres e Declarações
Relatório da Revisão Especial - Sem Ressalva 57
Declaração dos Diretores sobre as Demonstrações Financeiras 58
Declaração dos Diretores sobre o Relatório do Auditor Independente 60
ITR - Informações Trimestrais - 31/03/2021 - EMPREENDIMENTOS PAGUE MENOS S.A. Versão : 1
Dados da Empresa / Composição do Capital
Número de Ações(cid:10) Trimestre Atual(cid:10)
(Unidades) 31/03/2021
Do Capital Integralizado
Ordinárias 443.781.062
Preferenciais 0
Total 443.781.062
Em